# Classical additive decomposition (manual implementation)

This notebook shows how to manually perform additive decomposition.

Model:
$$Y_t = T_t + S_t + R_t$$


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def calculate_trend(series, window):
    series = pd.Series(series).reset_index(drop=True)
    n = len(series)
    trend = [np.nan] * n

    # Odd window -> MA
    if window % 2 == 1:

        half = (window - 1) // 2

        for i in range(n):

            if i < half or i >= n - half:
                continue

            trend[i] = series.iloc[
                i - half : i + half + 1
            ].mean()

    # Even window -> CMA
    else:

        ma = []

        for i in range(n - window + 1):

            ma.append(
                series.iloc[i : i + window].mean()
            )

        for i in range(len(ma) - 1):

            cma = (ma[i] + ma[i + 1]) / 2

            trend_index = i + window // 2

            trend[trend_index] = cma

    return trend


def additive_decomposition(df, value_col, period, window):

    result = df.copy()
    series = result[value_col].reset_index(drop=True)
    n = len(series)

    # Trend
    result['Trend'] = calculate_trend(series, window)

    # Remove trend
    result['Y_minus_T'] = (
        result[value_col] - result['Trend']
    )

    # Season number
    result['Season'] = [i % period for i in range(n)]

    # Seasonal effects
    seasonal = {}

    for season in range(period):

        values = result.loc[
            result['Season'] == season,
            'Y_minus_T'
        ].dropna()

        seasonal[season] = values.mean()

    # Normalize
    avg = np.mean(list(seasonal.values()))

    seasonal_norm = {}

    for season in seasonal:

        seasonal_norm[season] = (
            seasonal[season] - avg
        )

    # Assign seasonality
    result['Seasonality'] = [
        seasonal_norm[s]
        for s in result['Season']
    ]

    # Residuals
    result['Residual'] = (
        result[value_col]
        - result['Trend']
        - result['Seasonality']
    )

    return result


## Example 1 - odd window (MA)

In [ ]:
df1 = pd.DataFrame({
    'Sales': [
        100,108,117,
        106,114,123,
        112,120,129,
        118,126,135,
        124,132,141
    ]
})

result1 = additive_decomposition(
    df1,
    value_col='Sales',
    period=3,
    window=3
)

result1

## Example 2 - even window (CMA)

In [ ]:
df2 = pd.DataFrame({
    'Sales': [
        95,105,120,110,
        115,125,140,130,
        135,145,160,150
    ]
})

result2 = additive_decomposition(
    df2,
    value_col='Sales',
    period=4,
    window=4
)

result2